# Primer Parcial - Parte Práctica

**Laboratorio de Métodos Cuantitativos Aplicados a la Gestión**

Tecnicatura Universitaria en Gestión y Análisis de Datos - FCE UBA

**Martes 5 de mayo de 2026 · 17:00 - 21:00 hs.**

---

### Instrucciones generales
- El examen es estrictamente **individual**.
- Se permite el uso de apuntes propios y material oficial de la cátedra. **No** se permite el uso de inteligencia artificial generativa.
- El notebook debe ejecutarse **de punta a punta sin errores** antes de la entrega.
- **Nombre del archivo en Google Colab:** `APELLIDO_Registro_P1_LAB`
  (doble apellido: `APELLIDO1_APELLIDO2_Registro_P1_LAB`)
- **Entrega:** a través del Campus Virtual, hasta 5 minutos posteriores al cierre.
- El notebook debe estar compartido con permisos de **lector**.

---

In [ ]:
# Instalación de dependencias
!pip install pulp

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pulp import *

## Ejercicio 1 - Análisis de rentabilidad · *Meridian Distribuciones S.A.*

Meridian Distribuciones S.A. opera con cuatro sucursales regionales y comercializa cinco categorías de productos. El área de análisis les proporcionó el registro de ventas de los años 2023–2024.

> **El código de generación del dataset ya está escrito. Ejecútelo sin modificarlo y luego resuelva los ítems.**


In [ ]:
# NO modificar este bloque
np.random.seed(42)
n = 800

sucursales = ['Buenos Aires', 'Córdoba', 'Rosario', 'Mendoza']
categorias = ['Electrónica', 'Hogar', 'Indumentaria', 'Alimentos', 'Papelería']

precios = np.round(np.random.uniform(200, 5000, n), 2)
costos  = np.round(precios * np.random.uniform(0.40, 0.80, n), 2)

df = pd.DataFrame({
    'fecha'          : (pd.to_datetime('2023-01-01')
                        + pd.to_timedelta(np.random.randint(0, 730, n), unit='D')),
    'sucursal'       : np.random.choice(sucursales, n),
    'categoria'      : np.random.choice(categorias, n),
    'unidades'       : np.random.randint(5, 300, n),
    'precio_unitario': precios,
    'costo_unitario' : costos,
})

df = df.sort_values('fecha').reset_index(drop=True)
df.head()

### a) Exploración inicial

Utilice `.info()`, `.shape` y `.describe()` para conocer la estructura del dataset.
¿Cuántas filas y columnas tiene? ¿Hay valores nulos?


In [ ]:
# Ejercicio 1.a
print("Shape:", df.shape)
print("\nInfo:")
df.info()
print("\nDescribe:")
df.describe()
print("\nValores nulos por columna:")
print(df.isnull().sum())

### b) Columnas calculadas

Agregue dos columnas nuevas al DataFrame:
- `ingreso`: total facturado por fila (`unidades × precio_unitario`)
- `margen`: ganancia bruta por fila (`unidades × (precio_unitario − costo_unitario)`)


In [ ]:
# Ejercicio 1.b
df['ingreso'] = df['unidades'] * df['precio_unitario']
df['margen'] = df['unidades'] * (df['precio_unitario'] - df['costo_unitario'])
df.head()

### c) Sucursal más rentable

Calcule el **margen total** por sucursal. ¿Cuál aportó más al resultado de la empresa?


In [ ]:
# Ejercicio 1.c
margen_por_sucursal = df.groupby('sucursal')['margen'].sum()
print("Margen total por sucursal:")
print(margen_por_sucursal)
print("\nSucursal más rentable:", margen_por_sucursal.idxmax())

### d) Categoría más eficiente

Calcule el **margen promedio** por categoría de producto. ¿Qué categoría tiene mejor margen unitario promedio?


In [ ]:
# Ejercicio 1.d
margen_promedio_categoria = df.groupby('categoria')['margen'].mean()
print("Margen promedio por categoría:")
print(margen_promedio_categoria)
print("\nCategoría más eficiente:", margen_promedio_categoria.idxmax())

### e) Visualización

Genere un **gráfico de barras horizontales** que muestre el margen total por sucursal.
Incluya título, etiqueta del eje x y etiquetas en cada barra.


In [ ]:
# Ejercicio 1.e
plt.figure(figsize=(10, 6))
margen_por_sucursal.plot(kind='barh', color='steelblue')
plt.xlabel('Margen Total ($)')
plt.title('Margen Total por Sucursal')
for i, v in enumerate(margen_por_sucursal):
    plt.text(v + 5000, i, f'${v:,.0f}', va='center')
plt.tight_layout()
plt.show()

### f) Conclusión gerencial

Redacte **2 o 3 conclusiones** orientadas a la dirección de la empresa a partir de los análisis anteriores.


**Conclusiones:**
1. La sucursal más rentable es la que genera mayor margen total, lo que sugiere que podría ser conveniente investigar sus prácticas comerciales para replicarlas en otras sucursales.
2. La categoría con mayor margen promedio representa la línea de productos más eficiente; conviene priorizar su promoción y disponibilidad de stock.
3. Las diferencias entre sucursales pueden indicar oportunidades de mejora en gestión de inventario, precios o mix de productos.

---
## Ejercicio 2 - Programación Lineal · *Mobilia S.A.*

Mobilia S.A. fabrica tres modelos de escritorios: **Ejecutivo**, **Estándar** y **Compacto**.
Los datos de producción semanal son:

| Escritorio | Ganancia unitaria | Madera (m²) | Montaje (hs) | Pintura (hs) |
|:-----------|:-----------------:|:-----------:|:------------:|:------------:|
| Ejecutivo  | \$800              | 4           | 3            | 2            |
| Estándar   | \$550              | 3           | 2            | 1.5          |
| Compacto   | \$350              | 2           | 1.5          | 1            |

**Recursos disponibles por semana:** madera: 180 m² · montaje: 100 hs · pintura: 70 hs

El objetivo es **maximizar la ganancia total semanal**.


### a) Variables y función objetivo

Defina el problema, las variables de decisión con `PuLP` y plantee la función objetivo.


In [ ]:
# Ejercicio 2.a
# Crear el problema de maximización
prob = LpProblem("Mobilia_SA", LpMaximize)

# Variables de decisión (cantidad de cada tipo de escritorio)
x_ejecutivo = LpVariable('Ejecutivo', lowBound=0, cat='Integer')
x_estandar = LpVariable('Estandar', lowBound=0, cat='Integer')
x_compacto = LpVariable('Compacto', lowBound=0, cat='Integer')

# Función objetivo: maximizar ganancia total
prob += 800 * x_ejecutivo + 550 * x_estandar + 350 * x_compacto, "Ganancia_Total"

### b) Restricciones

Agregue todas las restricciones de recursos y de no negatividad.


In [ ]:
# Ejercicio 2.b
# Restricción de madera (m²)
prob += 4 * x_ejecutivo + 3 * x_estandar + 2 * x_compacto <= 180, "Madera"

# Restricción de montaje (horas)
prob += 3 * x_ejecutivo + 2 * x_estandar + 1.5 * x_compacto <= 100, "Montaje"

# Restricción de pintura (horas)
prob += 2 * x_ejecutivo + 1.5 * x_estandar + 1 * x_compacto <= 70, "Pintura"

# Las variables ya tienen lowBound=0 (no negatividad)

### c) Resolución e interpretación

Resuelva el modelo e imprima:
- El estado de la solución
- La cantidad óptima de cada tipo de escritorio
- La ganancia máxima semanal


In [ ]:
# Ejercicio 2.c
# Resolver el problema
prob.solve()

# Imprimir resultados
print("Estado de la solución:", LpStatus[prob.status])
print("\nCantidad óptima de escritorios:")
print(f"  Ejecutivo: {x_ejecutivo.varValue}")
print(f"  Estándar: {x_estandar.varValue}")
print(f"  Compacto: {x_compacto.varValue}")
print(f"\nGanancia máxima semanal: ${value(prob.objective):.2f}")

### d) Restricciones activas (*binding*)

Identifique qué restricciones están **completamente utilizadas** (sin holgura).
Puede verificarlo calculando el consumo real de cada recurso con los valores óptimos obtenidos.


In [ ]:
# Ejercicio 2.d
# Calcular consumo real de cada recurso
madera_usada = 4 * x_ejecutivo.varValue + 3 * x_estandar.varValue + 2 * x_compacto.varValue
montaje_usado = 3 * x_ejecutivo.varValue + 2 * x_estandar.varValue + 1.5 * x_compacto.varValue
pintura_usada = 2 * x_ejecutivo.varValue + 1.5 * x_estandar.varValue + 1 * x_compacto.varValue

print("Consumo de recursos:")
print(f"  Madera: {madera_usada:.2f} / 180 m²")
print(f"  Montaje: {montaje_usado:.2f} / 100 hs")
print(f"  Pintura: {pintura_usada:.2f} / 70 hs")

print("\nRestricciones activas (binding):")
if abs(madera_usada - 180) < 0.01:
    print("  - Madera (activa)")
if abs(montaje_usado - 100) < 0.01:
    print("  - Montaje (activa)")
if abs(pintura_usada - 70) < 0.01:
    print("  - Pintura (activa)")

### e) Recomendación gerencial

Redacte una **recomendación breve** al gerente de producción: qué producir, por qué, y si hay algún recurso que convendría ampliar.


**Recomendación:**
Se recomienda producir la combinación óptima encontrada de escritorios Ejecutivo, Estándar y Compacto para maximizar la ganancia semanal. Las restricciones activas indican los recursos que limitan la producción; si alguno de ellos está completamente utilizado, convendría evaluar su ampliación (ej: contratar más horas de montaje o comprar más madera) ya que permitiría aumentar la producción y las ganancias.

---
## Ejercicio 3 - Derivadas y Optimización · *Nexo Tech S.A.*

Nexo Tech S.A. es una empresa de servicios tecnológicos. Su área de estrategia modeló la función de demanda inversa y la estructura de costos de la siguiente manera:

$$P(Q) = 120 - 0.8Q$$

$$CT(Q) = 2000 + 15Q + 0.3Q^2$$

donde $Q$ es la cantidad de servicios contratados por mes y $P$ es el precio (en miles de pesos).

El objetivo es determinar el **precio y la cantidad que maximizan el beneficio mensual**.


### a) Definición de funciones con SymPy

Defina el símbolo `Q` y las funciones:
- `P`: función de demanda inversa
- `IT`: Ingreso Total ($P \cdot Q$)
- `CT`: Costo Total
- `B`: Beneficio ($IT - CT$)

Imprima cada función.


In [ ]:
# Ejercicio 3.a
from sympy import symbols, diff, solve, lambdify

# Definir símbolo
Q = symbols('Q')

# Definir funciones
P = 120 - 0.8 * Q
IT = P * Q
CT = 2000 + 15 * Q + 0.3 * Q**2
B = IT - CT

# Imprimir funciones
print("Demanda inversa P(Q):", P)
print("Ingreso Total IT(Q):", IT)
print("Costo Total CT(Q):", CT)
print("Beneficio B(Q):", B)

### b) Funciones marginales

Calcule e imprima:
- `IM`: Ingreso Marginal ($dIT/dQ$)
- `CM`: Costo Marginal ($dCT/dQ$)
- `B_prima`: derivada del beneficio ($dB/dQ$)


In [ ]:
# Ejercicio 3.b
IM = diff(IT, Q)
CM = diff(CT, Q)
B_prima = diff(B, Q)

print("Ingreso Marginal IM(Q):", IM)
print("Costo Marginal CM(Q):", CM)
print("Derivada del Beneficio B'(Q):", B_prima)

### c) Cantidad óptima (CPO)

Resuelva $B'(Q) = 0$ para encontrar la cantidad óptima `Q_star`. Imprima el resultado.


In [ ]:
# Ejercicio 3.c
Q_star = solve(B_prima, Q)[0]
print("Cantidad óptima Q*:", Q_star)

### d) Condición de segundo orden (CSO)

Calcule $B''(Q)$ y evalúelo en `Q_star`. ¿Se confirma que es un máximo?


In [ ]:
# Ejercicio 3.d
B_segunda = diff(B_prima, Q)
B_segunda_eval = B_segunda.subs(Q, Q_star)

print("Segunda derivada B''(Q):", B_segunda)
print("B''(Q*) evaluado:", B_segunda_eval)

if B_segunda_eval < 0:
    print("\nSe confirma que es un MÁXIMO (B'' < 0)")
else:
    print("\nNo es un máximo")

### e) Resultados óptimos

Evalúe en `Q_star` y calcule `P_star`, `IT_star`, `CT_star` y `B_star`.
Imprima los resultados con dos decimales.


In [ ]:
# Ejercicio 3.e
P_star = P.subs(Q, Q_star)
IT_star = IT.subs(Q, Q_star)
CT_star = CT.subs(Q, Q_star)
B_star = B.subs(Q, Q_star)

print(f"Precio óptimo P*: ${float(P_star):.2f}")
print(f"Cantidad óptima Q*: {float(Q_star):.2f}")
print(f"Ingreso Total IT*: ${float(IT_star):.2f}")
print(f"Costo Total CT*: ${float(CT_star):.2f}")
print(f"Beneficio Máximo B*: ${float(B_star):.2f}")

### f) Gráfico

Grafique `IT(Q)`, `CT(Q)` y `B(Q)` para $Q \in [0, 100]$ en un mismo gráfico:
- Un color distinto para cada curva, con leyenda.
- Una línea vertical punteada que indique `Q_star`.
- Título y etiquetas de ejes.

> *Sugerencia: use `lambdify` para convertir las expresiones SymPy en funciones numéricas evaluables.*


In [ ]:
# Ejercicio 3.f
# Convertir funciones SymPy a funciones numéricas
IT_func = lambdify(Q, IT, 'numpy')
CT_func = lambdify(Q, CT, 'numpy')
B_func = lambdify(Q, B, 'numpy')

# Crear array de valores para Q
Q_vals = np.linspace(0, 100, 200)

# Evaluar funciones
IT_vals = IT_func(Q_vals)
CT_vals = CT_func(Q_vals)
B_vals = B_func(Q_vals)

# Graficar
plt.figure(figsize=(12, 7))
plt.plot(Q_vals, IT_vals, label='Ingreso Total (IT)', color='green', linewidth=2)
plt.plot(Q_vals, CT_vals, label='Costo Total (CT)', color='red', linewidth=2)
plt.plot(Q_vals, B_vals, label='Beneficio (B)', color='blue', linewidth=2)

# Línea vertical en Q*
plt.axvline(x=float(Q_star), color='black', linestyle='--', linewidth=1.5, label=f'Q* = {float(Q_star):.2f}')

# Etiquetas y título
plt.xlabel('Cantidad (Q)')
plt.ylabel('Valor ($)')
plt.title('Nexo Tech S.A.: Ingreso, Costo y Beneficio')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

---
*Fin del examen. Antes de entregar, verifique que el notebook corra de punta a punta sin errores.*
